<a href="https://colab.research.google.com/github/ednnaldodev/Modulo4_PRF.ipynb-/blob/main/Modulo4_PRF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
import unicodedata
# Vamos conectar ao Google Drive pela praticidade!
from google.colab import drive
drive.mount ('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

In [ ]:
# Estrutura padrão do projeto
RAIZ = Path ("/content/drive/MyDrive/Colab Notebooks/prf_2025")
PASTAS = [
"dados_brutos", "dados_tratados",
"notebooks", "sql", "dashboards",
"relatorios", "apresentacao", "logs"
]
for pasta in PASTAS:
  (RAIZ / pasta).mkdir(
parents=True, exist_ok=True)
print("Pastas verificadas/criadas:")
for pasta in PASTAS:
  print("-", RAIZ / pasta)

Pastas verificadas/criadas:
- /content/drive/MyDrive/Colab Notebooks/prf_2025/dados_brutos
- /content/drive/MyDrive/Colab Notebooks/prf_2025/dados_tratados
- /content/drive/MyDrive/Colab Notebooks/prf_2025/notebooks
- /content/drive/MyDrive/Colab Notebooks/prf_2025/sql
- /content/drive/MyDrive/Colab Notebooks/prf_2025/dashboards
- /content/drive/MyDrive/Colab Notebooks/prf_2025/relatorios
- /content/drive/MyDrive/Colab Notebooks/prf_2025/apresentacao
- /content/drive/MyDrive/Colab Notebooks/prf_2025/logs


In [ ]:
# Parâmetros do projeto
ARQUIVO_BRUTO = Path("dados_brutos/acidentes2025.csv")
ARQUIVO_BASE_ANALITICA = Path ("dados_tratados/base_analitica_completa.csv")
ARQUIVO_BASE_MODELAVEL = Path ("dados_tratados/base_modelavel_preliminar.csv")
ARQUIVO_DICIONARIO = Path ("dados_tratados/dicionario_variaveis_modulo4.csv")
ARQUIVO_DECISOES = Path ("logs/decisoes_tratamento_modulo4.md")
ARQUIVO_README = Path ("README.md")
SEPARADOR = ";"
ENCODING_ENTRADA = "latin1"
ENCODING_SAIDA = "utf-8-sig"

In [ ]:
# Leitura CSV da PRF (com Fallback)
def ler_csv_prf(caminho, sep=";", encodings=("utf-8-sig", "utf-8", "latin1")):
  ultimo_erro = None
  for enc in encodings:
    try:
      print(f"Tentando leitura com encodin={enc}...")
      return pd.read_csv(caminho, sep=sep, encoding=enc, low_memory=False)
    except Exception as erro:
      ultimo_erro = erro
      print(f"Falhou com {enc}: {erro}")
    raise ultimo_erro

In [ ]:
df = ler_csv_prf(RAIZ / ARQUIVO_BRUTO, sep=SEPARADOR)
df.head(5)

Tentando leitura com encodin=utf-8-sig...


,id,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,tipo_acidente,classificacao_acidente,fase_dia,sentido_via,condicao_metereologica,tipo_pista,tracado_via,uso_solo,pessoas,mortos,feridos_leves,feridos_graves,ilesos,ignorados,feridos,veiculos,latitude,longitude,regional,delegacia,uop
0,652493,2025-01-01,quarta-feira,06:20:00,SP,116,225,GUARULHOS,Reação tardia ou ineficiente do condutor,Tombamento,Com Vítimas Feridas,Pleno dia,Decrescente,Céu Claro,Múltipla,Reta;Declive,Sim,2,0,1,0,0,1,1,2,"-23,48586772","-46,54075317",SPRF-SP,DEL01-SP,UOP01-DEL01-SP
1,652519,2025-01-01,quarta-feira,07:50:00,CE,116,"546,2",PENAFORTE,Pista esburacada,Colisão frontal,NaN,Pleno dia,Crescente,Céu Claro,Simples,Reta,Não,6,1,1,0,1,4,1,6,"-7,812288","-39,08333306",SPRF-CE,DEL05-CE,UOP03-DEL05-CE
2,652522,2025-01-01,quarta-feira,08:45:00,PR,369,"88,2",CORNELIO PROCOPIO,Reação tardia ou ineficiente do condutor,Colisão traseira,Com Vítimas Feridas,Pleno dia,Crescente,Sol,Dupla,Reta;Aclive,Sim,5,0,3,0,2,0,3,2,"-23,182565","-50,637228",SPRF-PR,DEL07-PR,UOP05-DEL07-PR
3,652544,2025-01-01,quarta-feira,11:00:00,PR,116,74,CAMPINA GRANDE DO SUL,Reação tardia ou ineficiente do condutor,Saída de leito carroçável,Com Vítimas Feridas,Pleno dia,Crescente,Céu Claro,Dupla,Reta,Não,5,0,1,0,4,0,1,2,"-25,36517687","-49,04223028",SPRF-PR,DEL01-PR,UOP02-DEL01-PR
4,652549,2025-01-01,quarta-feira,09:30:00,MG,251,471,FRANCISCO SA,Velocidade Incompatível,Colisão frontal,Com Vítimas Feridas,Pleno dia,Decrescente,Chuva,Simples,Curva;Declive,Não,5,0,1,1,1,2,2,4,"-16,46801304","-43,43121303",SPRF-MG,DEL12-MG,UOP01-DEL12-MG


In [ ]:
# Padronizaçao dos nome das colunas
def normalizar_nome_coluna(nome):
  nome = str(nome).strip().lower()
  nome = unicodedata.normalize("NFKD", nome).encode("ascii", "ignore").decode("utf-8")
  nome = nome.replace(" ", "_").replace("-", "_").replace("/", "_")
  while "__" in nome:
    nome = nome.replace("__", "_")
  return nome.strip("_")
df.columns = [normalizar_nome_coluna(c) for c in df.columns]

# Compatibilização de grafias possíveis

df = df.rename(columns={"condicao_metereologica": "condicao_meteorologica"})
print(df.columns.tolist())

['id', 'data_inversa', 'dia_semana', 'horario', 'uf', 'br', 'km', 'municipio', 'causa_acidente', 'tipo_acidente', 'classificacao_acidente', 'fase_dia', 'sentido_via', 'condicao_meteorologica', 'tipo_pista', 'tracado_via', 'uso_solo', 'pessoas', 'mortos', 'feridos_leves', 'feridos_graves', 'ilesos', 'ignorados', 'feridos', 'veiculos', 'latitude', 'longitude', 'regional', 'delegacia', 'uop']


In [ ]:
# Compatibilização de grafias possíveis
df = df.rename(columns={"condicao_metereologica": "condicao_meteorologica"})
print(df.columns.tolist())

['id', 'data_inversa', 'dia_semana', 'horario', 'uf', 'br', 'km', 'municipio', 'causa_acidente', 'tipo_acidente', 'classificacao_acidente', 'fase_dia', 'sentido_via', 'condicao_meteorologica', 'tipo_pista', 'tracado_via', 'uso_solo', 'pessoas', 'mortos', 'feridos_leves', 'feridos_graves', 'ilesos', 'ignorados', 'feridos', 'veiculos', 'latitude', 'longitude', 'regional', 'delegacia', 'uop']


In [ ]:
colunas_esperadas = [
"data_inversa", "dia_semana", "horario", "uf", "br", "municipio",
"causa_acidente", "tipo_acidente", "classificacao_acidente",
"fase_dia", "condicao_meteorologica", "tipo_pista", "tracado_via",
"uso_solo", "pessoas", "mortos", "feridos_leves",
"feridos_graves", "feridos", "veiculos"
]

faltantes = [c for c in colunas_esperadas if c not in df.columns]
print("Colunas faltantes:", faltantes)

if faltantes:
  print("Atenção: ajuste nomes ou confirme o dicionário oficial da PRF usado no arquivo. ")

Colunas faltantes: []


In [ ]:
# Tipos de dados e memória utilizada
df.info(memory_usage="deep")
resumo_tipos = (
df.dtypes.astype(str)
.value_counts()
.rename_axis("tipo")
.reset_index(name="qtd_colunas")
)
display(resumo_tipos)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72529 entries, 0 to 72528
Data columns (total 30 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   id                      72529 non-null  int64 
 1   data_inversa            72529 non-null  object
 2   dia_semana              72529 non-null  object
 3   horario                 72529 non-null  object
 4   uf                      72529 non-null  object
 5   br                      72529 non-null  int64 
 6   km                      72529 non-null  object
 7   municipio               72529 non-null  object
 8   causa_acidente          72529 non-null  object
 9   tipo_acidente           72529 non-null  object
 10  classificacao_acidente  72528 non-null  object
 11  fase_dia                72529 non-null  object
 12  sentido_via             72529 non-null  object
 13  condicao_meteorologica  72529 non-null  object
 14  tipo_pista              72529 non-null  object
 15  tr

,tipo,qtd_colunas
0,object,20
1,int64,10


In [ ]:
# Diagnóstico de valores ausentes
nulos = pd.DataFrame({
"qtd_nulos": df.isna().sum(),
"perc_nulos": df.isna().mean() * 100
}).sort_values(
"perc_nulos", ascending=False)
display(nulos[nulos["qtd_nulos"] > 0])

,qtd_nulos,perc_nulos
uop,38,0.052393
delegacia,22,0.030333
regional,2,0.002758
classificacao_acidente,1,0.001379


In [ ]:
# Diagnóstico e remoção de duplicidades
qtd_duplicadas = df.duplicated().sum()

print("Duplicidades exatas:", qtd_duplicadas)
if qtd_duplicadas > 0:
  df = df.drop_duplicates().copy()

print("Duplicidades removidas.")
print("Nova dimensão:", df.shape)

Duplicidades exatas: 0
Duplicidades removidas.
Nova dimensão: (72529, 30)


In [ ]:
# Cardinalidade das variáveis categóricas
categoricas = df.select_dtypes(
include="object").columns
cardinalidade = (
df[categoricas]
.nunique(dropna=True)
.sort_values(ascending=False)
.reset_index()
)
cardinalidade.columns = [
  "variavel","qtd_categorias"]
display(cardinalidade.head(30))

,variavel,qtd_categorias
0,latitude,69294
1,longitude,69237
2,km,7655
3,municipio,1844
4,horario,1412
5,tracado_via,605
6,uop,395
7,data_inversa,365
8,delegacia,153
9,causa_acidente,69


In [ ]:
colunas_numericas = [
"br","km","pessoas","mortos","feridos",
"feridos_leves","feridos_graves",
"ilesos","ignorados","veiculos"
]

for coluna in colunas_numericas:
 if coluna in df.columns:
  df[coluna] = pd.to_numeric(df[coluna], errors="coerce")

print(df[[c for c in colunas_numericas if c in df.columns]].dtypes)

br                  int64
km                float64
pessoas             int64
mortos              int64
feridos             int64
feridos_leves       int64
feridos_graves      int64
ilesos              int64
ignorados           int64
veiculos            int64
dtype: object
